# Day 03：流程控制与多轮交互路由 —— 智枢平台 CLI 调度中心

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》实战第 3 天！我是 Tech Lead 张工。前两天我们已经搭好了员工数字名片（Day 01）和 Prompt 脱敏引擎（Day 02）。今天我们要用 Python 的三大流程控制武器——**嵌套 `if/elif/else`**、**`for` 循环（配合 `continue`/`break`）** 以及 **`while` 循环**，把智枢平台升级为一个能做权限分级、批量清洗 Prompt 并持续响应指令的交互式调度中心！

*(注：为保证自动化教学验证可无阻塞运行，本 Notebook 使用模拟输入列表代替 `input()` 交互)*

## 1. 第一步：嵌套条件判断 (`if / elif / else`) —— 智枢平台权限分级校验
在企业平台中，不同员工的大模型算力权限必须严格分层：
- 若 `llm_enabled` 为 `False`，无论工龄和预算多高，一律返回 `"DENIED"`；
- 若 `llm_enabled` 为 `True`，再进一步判断：
  - 工龄 `>= 3` 年 **且** 预算 `>= 50.0` → `"VIP"`
  - 预算 `> 0` 且（工龄 `>= 1` 年 **或** 预算 `>= 10.0`）→ `"STANDARD"`
  - 其他情况 → `"DENIED"`

In [1]:
def check_access_level(years_exp: int, llm_enabled: bool, budget: float) -> str:
    """使用嵌套 if/elif/else 判断员工的智枢平台访问权限等级。"""
    # 外层 if：先检查是否已开通大模型总权限开关
    if llm_enabled:
        # 内层 if：同时满足资深工龄与高额预算，授予 VIP 权限
        if years_exp >= 3 and budget >= 50.0:
            return "VIP"
        # 内层 elif：预算为正，且满足基本工龄或基本预算门槛，授予标准权限
        elif budget > 0 and (years_exp >= 1 or budget >= 10.0):
            return "STANDARD"
        # 内层 else：其余情况拒绝访问
        else:
            return "DENIED"
    else:
        # 未开通权限开关，直接拒绝
        return "DENIED"

# 使用多组测试数据验证边界条件
test_cases = [
    ("资深架构师陈默", 3, True, 60.0),
    ("转行学员小王", 0, True, 15.0),
    ("未授权访客", 5, False, 100.0),
    ("预算归零账号", 2, True, 0.0),
]
for name, exp, enabled, bgt in test_cases:
    level = check_access_level(exp, enabled, bgt)
    print(f"{name:<10} | 工龄={exp}年 | 权限={str(enabled):<5} | 预算=¥{bgt:<5.1f} -> 等级: {level}")

资深架构师陈默    | 工龄=3年 | 权限=True  | 预算=¥60.0  -> 等级: VIP
转行学员小王     | 工龄=0年 | 权限=True  | 预算=¥15.0  -> 等级: STANDARD
未授权访客      | 工龄=5年 | 权限=False | 预算=¥100.0 -> 等级: DENIED
预算归零账号     | 工龄=2年 | 权限=True  | 预算=¥0.0   -> 等级: DENIED


## 2. 第二步：`for` 循环与 `continue` / `break` —— 批量 Prompt 清洗引擎
在实际业务中，用户一次可能提交多条待处理的 Prompt：
- 遇到空白行时，使用 `continue` 跳过本轮，避免浪费大模型 Token；
- 遇到停止词 `"STOP"` 时，使用 `break` 立即终止批处理队列，防止后续无效内容继续消耗资源。

In [2]:
import re

def batch_clean_prompts(prompts: list[str], stop_keyword: str = "STOP") -> list[str]:
    """批量清洗 Prompt 列表：遇空跳过(continue)，遇停止词终止(break)。"""
    cleaned_list: list[str] = []                  # 用于收集有效且清洗后的 Prompt
    sensitive_words = ["密码", "API_KEY", "sk-"]  # 模拟 Day 02 的敏感词表

    for raw_prompt in prompts:                  # 遍历原始列表中的每一条 Prompt
        stripped = raw_prompt.strip()           # 去除首尾空格、制表符和换行符
        if not stripped:                        # 若去除空白后为空字符串
            continue                            # 跳过当前循环，继续处理下一条
        if stripped == stop_keyword:            # 若遇到停止关键词（如 'STOP'）
            break                               # 立即跳出整个 for 循环，不再处理后续元素

        # 折叠内部多余空白字符为单个空格
        normalized = re.sub(r"\s+", " ", stripped)
        # 对敏感词进行脱敏替换
        for word in sensitive_words:
            normalized = normalized.replace(word, "***")
        cleaned_list.append(normalized)         # 将清洗结果加入输出列表

    return cleaned_list

raw_queue = [
    "   请帮我总结   今日会议纪要  ",
    "",
    "   \t  ",
    "排查我的 API_KEY 是否泄露",
    "STOP",
    "这条消息位于 STOP 之后，不会被处理",
]
result = batch_clean_prompts(raw_queue)
print(f"共收集到 {len(result)} 条有效 Prompt:")
for idx, item in enumerate(result, start=1):
    print(f"  {idx}. {item}")

共收集到 2 条有效 Prompt:
  1. 请帮我总结 今日会议纪要
  2. 排查我的 *** 是否泄露


## 3. 第三步：经典嵌套 `for` 循环 —— 9×9 乘法口诀表
双层 `for` 循环是所有后端与 AI 工程师必须掌握的基本功（后续做多模型×多 Prompt 评测矩阵时也是同样的双层循环结构）。

In [3]:
def generate_multiplication_table(n: int = 9) -> str:
    """使用嵌套 for 循环生成 n x n 下三角乘法口诀表。"""
    rows: list[str] = []                        # 存放每一行的字符串
    for i in range(1, n + 1):                   # 外层循环：控制行号 i（从 1 到 n）
        cols: list[str] = []                    # 存放当前行的所有算式
        for j in range(1, i + 1):               # 内层循环：控制列号 j（从 1 到 i）
            cols.append(f"{j}×{i}={i * j:<2}")   # 左对齐占 2 位，保证列对齐
        rows.append("  ".join(cols).rstrip())    # 用两个空格拼接当前行
    return "\n".join(rows)                     # 用换行符拼接所有行

print(generate_multiplication_table(9))

1×1=1
1×2=2   2×2=4
1×3=3   2×3=6   3×3=9
1×4=4   2×4=8   3×4=12  4×4=16
1×5=5   2×5=10  3×5=15  4×5=20  5×5=25
1×6=6   2×6=12  3×6=18  4×6=24  5×6=30  6×6=36
1×7=7   2×7=14  3×7=21  4×7=28  5×7=35  6×7=42  7×7=49
1×8=8   2×8=16  3×8=24  4×8=32  5×8=40  6×8=48  7×8=56  8×8=64
1×9=9   2×9=18  3×9=27  4×9=36  5×9=45  6×9=54  7×9=63  8×9=72  9×9=81


## 4. 第四步：`while` 循环与菜单路由分发 —— `run_menu_dispatch`
真实 CLI 程序使用 `while True` 持续等待用户选择。在 Notebook 中，我们用一个预设指令序列 `simulated_choices = ["1", "2", "99", "0"]` 来模拟多轮交互，并在收到 `"0"` 时通过 `break` 优雅退出。

In [4]:
def run_menu_dispatch(choice: str, context: dict) -> str:
    """根据菜单编号分发到对应业务功能。"""
    normalized = choice.strip()
    if normalized == "1":
        level = check_access_level(
            context["years_exp"],
            context["llm_enabled"],
            context["budget"],
        )
        return f"[名片核验] 员工 {context['name']} 当前权限等级为: {level}"
    elif normalized == "2":
        cleaned = batch_clean_prompts(context["prompts"])
        return f"[批量清洗] 成功处理 {len(cleaned)} 条有效 Prompt: {cleaned}"
    elif normalized == "3":
        return generate_multiplication_table(context.get("n", 4))
    elif normalized == "0":
        return "[系统退出] 感谢使用智枢企业级智能体平台 (v0.1.0-day03)！"
    else:
        return f"[提示] 无效指令 '{choice}'，请输入 0-3 之间的选项。"

# 模拟多轮交互上下文与指令流（无 input() 阻塞）
sim_context = {
    "name": "陈默",
    "years_exp": 3,
    "llm_enabled": True,
    "budget": 60.0,
    "prompts": ["  生成周报 ", "", "STOP", "忽略"],
}
simulated_choices = ["1", "2", "99", "0"]
idx = 0

while idx < len(simulated_choices):             # 模拟 while True 多轮调度循环
    user_choice = simulated_choices[idx]
    idx += 1
    print(f">>> 用户输入指令: {user_choice}")
    output = run_menu_dispatch(user_choice, sim_context)
    print(output)
    if user_choice == "0":                      # 收到退出指令，break 跳出循环
        break

>>> 用户输入指令: 1
[名片核验] 员工 陈默 当前权限等级为: VIP
>>> 用户输入指令: 2
[批量清洗] 成功处理 1 条有效 Prompt: ['生成周报']
>>> 用户输入指令: 99
[提示] 无效指令 '99'，请输入 0-3 之间的选项。
>>> 用户输入指令: 0
[系统退出] 感谢使用智枢企业级智能体平台 (v0.1.0-day03)！
